# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# ============================================================
# PASO 1. CONFIGURAR GROQ DE FORMA SEGURA CON COLAB SECRETS
# ============================================================
#
# Esta celda instala el cliente de Groq y recupera la API key
# desde el apartado "Secrets" de Google Colab.
#
# ¿Por qué usamos Secrets?
# Porque NO debemos escribir la API key directamente en el código.
# Si compartimos el notebook, nuestra clave no queda expuesta.
#
# Antes de ejecutar:
# 1. Abre el ícono de llave en la barra lateral de Colab.
# 2. Crea un secreto llamado exactamente: GROQ_API_KEY
# 3. Pega tu API key de Groq.
# 4. Activa el acceso del notebook al secreto.

!pip install groq --quiet

from groq import Groq
from google.colab import userdata

# Recuperamos la clave guardada y creamos el cliente.
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

print("Cliente de Groq inicializado correctamente.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.4 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [2]:
# ============================================================
# PASO 2. PROMPT ZERO-SHOT
# ============================================================
#
# Zero-shot = pedir una tarea directamente, sin mostrar ejemplos.
#
# Aquí pedimos clasificar una reseña como:
# Positivo, Negativo o Mixto.
#
# Observa que solo damos la instrucción y la reseña.

prompt_zero_shot = (
    "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: "
    "'El envío llegó tarde pero el producto es excelente.' "
    "Respuesta muy breve y corta."
)

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_zero_shot
        }
    ]
)

print("Zero-shot:", response_zero.choices[0].message.content)

# Pregunta para el alumno:
# ¿La respuesta respeta exactamente una de las categorías pedidas?


Zero-shot: Mixto


In [3]:
# ============================================================
# PASO 3. PROMPT FEW-SHOT
# ============================================================
#
# Few-shot = mostrar unos pocos ejemplos de entrada y salida
# antes de pedir al modelo que resuelva el nuevo caso.
#
# En este ejemplo mostramos:
# - una reseña positiva;
# - una reseña negativa;
# y después dejamos una nueva reseña para clasificar.
#
# Los ejemplos ayudan a que el modelo entienda mejor el formato
# de respuesta esperado.

prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_few_shot
        }
    ]
)

print("Few-shot:", response_few.choices[0].message.content)

# Compara con la celda anterior:
# ¿El ejemplo few-shot acotó mejor la salida a una categoría esperada?


Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [4]:
# ============================================================
# PASO 4. EJEMPLO DE RAZONAMIENTO PASO A PASO
# ============================================================
#
# En esta sección del Hands On se pide al modelo que muestre
# el desarrollo antes de dar la respuesta final.
#
# El propósito didáctico es observar cómo cambia la respuesta
# cuando el prompt solicita explícitamente un procedimiento.

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": problema
        }
    ]
)

print(response_cot.choices[0].message.content)

# Lo importante aquí no es memorizar la respuesta.
# Observa cómo cambia el formato cuando el prompt exige desarrollo.


**Paso 1 – Entender el problema**

- Tren A: sale de la ciudad A con 80 km/h.  
- Tren B: sale 2 h después de A con 120 km/h.  
- Ambos trenes viajan en la misma dirección y hacia el mismo destino.

Queremos saber cuánto tiempo necesita el tren B para alcanzar al tren A.

---

**Paso 2 – Calcular la ventaja (distancia de adelanto) que tiene el tren A**

El tren A viaja durante 2 h antes de que el tren B salga.  
\[
\text{Distancia de ventaja} = \text{velocidad}_A \times \text{tiempo de adelanto}
= 80 \, \frac{\text{km}}{\text{h}} \times 2 \, \text{h}
= 160 \text{ km}.
\]

---

**Paso 3 – Determinar la velocidad relativa**

Para que el tren B alcance al tren A, debe cerrar esa distancia de 160 km a una velocidad que es la diferencia de sus velocidades:
\[
\text{velocidad relativa} = \text{velocidad}_B - \text{velocidad}_A
= 120 \, \frac{\text{km}}{\text{h}} - 80 \, \frac{\text{km}}{\text{h}}
= 40 \, \frac{\text{km}}{\text{h}}.
\]

---

**Paso 4 – Calcular el tiempo de persecución**

\[


### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [5]:
# ============================================================
# PASO 5. OBSERVAR EL LÍMITE DEL PROMPTING
# ============================================================
#
# Un prompt mejor redactado NO le entrega información nueva
# al modelo.
#
# Aquí preguntamos por un dato muy específico que el modelo
# no necesariamente tiene disponible.
#
# Si responde con seguridad sin contar con esa información,
# debemos considerar que podría estar inventando o alucinando.

prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_desconocido
        }
    ]
)

print(response_alucinacion.choices[0].message.content)

# Idea clave:
# Prompt engineering mejora cómo pedimos una respuesta.
# No agrega documentos ni hechos nuevos al modelo.


El equipo **CodeSprint** fue el ganador.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [6]:
# ============================================================
# PASO 6. INSTALAR SENTENCE-TRANSFORMERS
# ============================================================
#
# Ahora empieza la parte de RAG.
#
# sentence-transformers nos permitirá convertir texto en
# embeddings: vectores numéricos que representan significado.
#
# NumPy se usará para realizar operaciones matemáticas
# sobre esos vectores.

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

print("Librerías listas para trabajar con embeddings.")


Librerías listas para trabajar con embeddings.


In [7]:
# ============================================================
# PASO 7. CREAR LA BASE DE CONOCIMIENTO Y SUS EMBEDDINGS
# ============================================================
#
# Este Hands On utiliza tres fragmentos de políticas de tienda.
# Cada fragmento funciona como un pequeño documento.
#
# Después cargamos un modelo de embeddings multilingüe.
# encode(documentos) transforma cada texto en un vector.
#
# IMPORTANTE:
# Estos embeddings NO son respuestas.
# Son representaciones numéricas que luego podremos comparar.

modelo_embeddings = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",

    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",

    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)

print("Cantidad de fragmentos:", len(documentos))
print("Forma de la matriz de embeddings:", embeddings_documentos.shape)

# La primera dimensión corresponde al número de fragmentos.
# La segunda corresponde al tamaño del vector producido por el modelo.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cantidad de fragmentos: 3
Forma de la matriz de embeddings: (3, 384)


In [8]:
# ============================================================
# PASO 8. RECUPERAR EL FRAGMENTO MÁS RELEVANTE
# ============================================================
#
# Esta función recibe una pregunta y:
# 1. convierte la pregunta en un embedding;
# 2. compara ese embedding contra los embeddings de documentos;
# 3. identifica el índice con mayor similitud;
# 4. devuelve el fragmento correspondiente.
#
# NOTA TÉCNICA:
# El Hands On original usa np.dot(...) para calcular una puntuación
# entre los vectores. Aquí conservamos exactamente ese procedimiento.
#
# Más abajo, en el ejemplo del Challenge, mostraremos similitud coseno
# de forma explícita porque el enunciado del Challenge la solicita.

def buscar_fragmento(pregunta):

    # Convertimos la pregunta a vector.
    embedding_pregunta = modelo_embeddings.encode([pregunta])

    # Calculamos una puntuación entre la pregunta y cada documento.
    similitudes = np.dot(
        embeddings_documentos,
        embedding_pregunta.T
    ).flatten()

    # np.argmax(...) devuelve la posición del valor más alto.
    indice_mas_similar = np.argmax(similitudes)

    # Regresamos el texto almacenado en esa posición.
    return documentos[indice_mas_similar]


pregunta = "¿Puedo devolver algo que compré en oferta?"

fragmento = buscar_fragmento(pregunta)

print("Pregunta:", pregunta)
print("\nFragmento recuperado:")
print(fragmento)


Pregunta: ¿Puedo devolver algo que compré en oferta?

Fragmento recuperado:
Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [9]:
# ============================================================
# PASO 9. CONSTRUIR EL PROMPT CON CONTEXTO Y RESPONDER CON RAG
# ============================================================
#
# Ya recuperamos un fragmento relevante.
# Ahora lo agregamos al prompt como CONTEXTO.
#
# La instrucción "usando SOLO la siguiente política" busca que
# la respuesta quede anclada al fragmento recuperado.
#
# Este es el patrón esencial de este mini-RAG:
#
# pregunta
#    ↓
# recuperar fragmento
#    ↓
# colocar fragmento en el prompt
#    ↓
# enviar prompt al modelo
#    ↓
# generar respuesta basada en contexto

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda.
Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_rag
        }
    ]
)

print("Respuesta con RAG:")
print(response_rag.choices[0].message.content)

# Reflexión:
# ¿Qué información del fragmento recuperado aparece en la respuesta?


Respuesta con RAG:
No, los productos en oferta no se pueden devolver; solo pueden cambiarse de talla.


---
# **EJEMPLO DIDÁCTICO DE CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

> **Importante:** esta sección es un ejemplo de apoyo para explicar el Challenge al alumnado.  
> No sustituye las instrucciones exactas de la sección oficial del Challenge.

De acuerdo con el enunciado compartido, el Challenge busca demostrar tres competencias:

1. **Generación de embeddings:** vectorizar fragmentos de un documento propio con `sentence-transformers`.
2. **Recuperación de información:** calcular **similitud coseno** para encontrar el fragmento más relevante.
3. **Prompting con contexto:** comparar una respuesta **sin RAG** contra una respuesta **con RAG**.

La idea es reutilizar el patrón visto en el Hands On, pero ahora con una pequeña base de conocimiento propia.


## **Ejemplo que usaremos**

Imaginemos que una tienda tiene estas políticas internas:

- Política de cambios.
- Política de garantía.
- Política de envíos.

La pregunta será:

> **“Mi producto dejó de funcionar después de dos meses, ¿qué puedo hacer?”**

Primero veremos qué responde el modelo **sin darle las políticas**.  
Después buscaremos el fragmento más relevante y lo agregaremos al prompt para obtener la respuesta **con RAG**.


In [10]:
# ============================================================
# CHALLENGE - PASO 1. CREAR UN DOCUMENTO PROPIO POR FRAGMENTOS
# ============================================================
#
# En un sistema RAG real, un documento grande suele dividirse
# en fragmentos pequeños.
#
# Para este ejemplo didáctico usaremos tres fragmentos escritos
# directamente en una lista.

politicas_propias = [
    (
        "CAMBIOS: Los productos pueden cambiarse dentro de los primeros 15 días "
        "posteriores a la compra, siempre que estén sin uso y con empaque original."
    ),
    (
        "GARANTÍA: Los productos electrónicos cuentan con 6 meses de garantía por "
        "fallas de fabricación. La garantía no cubre golpes, humedad ni daños por mal uso."
    ),
    (
        "ENVÍOS: Los pedidos nacionales se entregan entre 3 y 5 días hábiles. "
        "Los retrasos provocados por eventos externos pueden extender el tiempo de entrega."
    )
]

print("Fragmentos cargados:", len(politicas_propias))

for i, fragmento in enumerate(politicas_propias, start=1):
    print(f"\nFragmento {i}:")
    print(fragmento)


Fragmentos cargados: 3

Fragmento 1:
CAMBIOS: Los productos pueden cambiarse dentro de los primeros 15 días posteriores a la compra, siempre que estén sin uso y con empaque original.

Fragmento 2:
GARANTÍA: Los productos electrónicos cuentan con 6 meses de garantía por fallas de fabricación. La garantía no cubre golpes, humedad ni daños por mal uso.

Fragmento 3:
ENVÍOS: Los pedidos nacionales se entregan entre 3 y 5 días hábiles. Los retrasos provocados por eventos externos pueden extender el tiempo de entrega.


In [11]:
# ============================================================
# CHALLENGE - PASO 2. GENERAR LOS EMBEDDINGS DE LOS FRAGMENTOS
# ============================================================
#
# Reutilizamos el mismo modelo de embeddings del Hands On.
#
# encode(...) convierte cada política en un vector.
# normalize_embeddings=True normaliza los vectores.
# Esto facilita utilizar similitud coseno de forma consistente.

embeddings_politicas = modelo_embeddings.encode(
    politicas_propias,
    normalize_embeddings=True
)

print("Forma de los embeddings:")
print(embeddings_politicas.shape)


Forma de los embeddings:
(3, 384)


In [12]:
# ============================================================
# CHALLENGE - PASO 3. HACER PRIMERO LA PREGUNTA SIN RAG
# ============================================================
#
# Esta será nuestra línea base.
# El modelo recibe únicamente la pregunta.
#
# Todavía NO le damos las políticas de nuestra tienda.

pregunta_challenge = (
    "Mi producto electrónico dejó de funcionar después de dos meses, "
    "¿qué puedo hacer?"
)

prompt_sin_rag = f"""Responde brevemente la siguiente pregunta de un cliente.

Pregunta: {pregunta_challenge}
"""

respuesta_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_sin_rag
        }
    ]
)

print("RESPUESTA SIN RAG:")
print(respuesta_sin_rag.choices[0].message.content)


RESPUESTA SIN RAG:
¡Lo sentimos mucho!  
1️⃣ Contacta a nuestro servicio de atención al cliente con tu número de pedido y el comprobante de compra.  
2️⃣ Si el producto está dentro del periodo de garantía, lo enviamos para reparación o reemplazo sin cargo.  
3️⃣ Si ya supera la garantía, te ofrecemos un descuento especial o un crédito para una nueva compra.  

Nuestro objetivo es resolverlo lo antes posible. ¡Gracias por confiar en nosotros!


In [13]:
# ============================================================
# CHALLENGE - PASO 4. CALCULAR SIMILITUD COSENO
# ============================================================
#
# El enunciado del Challenge pide explícitamente similitud coseno.
#
# Para hacerla visible de forma clara importamos cosine_similarity.
# Compararemos:
#
# embedding de la pregunta
#           vs.
# embeddings de todas las políticas
#
# El resultado será una puntuación por cada fragmento.

from sklearn.metrics.pairwise import cosine_similarity

embedding_pregunta = modelo_embeddings.encode(
    [pregunta_challenge],
    normalize_embeddings=True
)

similitudes = cosine_similarity(
    embedding_pregunta,
    embeddings_politicas
)[0]

print("Puntuaciones de similitud:")

for i, puntuacion in enumerate(similitudes, start=1):
    print(f"Fragmento {i}: {puntuacion:.4f}")


Puntuaciones de similitud:
Fragmento 1: 0.3623
Fragmento 2: 0.4847
Fragmento 3: 0.2130


In [14]:
# ============================================================
# CHALLENGE - PASO 5. RECUPERAR EL FRAGMENTO MÁS RELEVANTE
# ============================================================
#
# np.argmax(...) nos dice en qué posición está la puntuación mayor.
#
# Después usamos esa misma posición para recuperar el texto original.

indice_mejor = np.argmax(similitudes)

fragmento_relevante = politicas_propias[indice_mejor]

print("Fragmento recuperado:")
print(fragmento_relevante)

print("\nSimilitud coseno:")
print(f"{similitudes[indice_mejor]:.4f}")


Fragmento recuperado:
GARANTÍA: Los productos electrónicos cuentan con 6 meses de garantía por fallas de fabricación. La garantía no cubre golpes, humedad ni daños por mal uso.

Similitud coseno:
0.4847


In [15]:
# ============================================================
# CHALLENGE - PASO 6. CONSTRUIR EL PROMPT CON CONTEXTO
# ============================================================
#
# Ahora sí usamos RAG:
# tomamos el fragmento recuperado y se lo entregamos al modelo.
#
# Observa la instrucción:
# "usa exclusivamente la política proporcionada".
#
# Esto busca reducir respuestas inventadas fuera del documento.

prompt_con_rag = f"""Eres un asistente de atención al cliente.

Responde usando exclusivamente la política proporcionada.
Si la política no contiene información suficiente, indícalo claramente.

POLÍTICA RECUPERADA:
{fragmento_relevante}

PREGUNTA DEL CLIENTE:
{pregunta_challenge}

Respuesta breve:
"""

respuesta_con_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_con_rag
        }
    ]
)

print("RESPUESTA CON RAG:")
print(respuesta_con_rag.choices[0].message.content)


RESPUESTA CON RAG:
Tu producto está dentro del plazo de garantía de 6 meses por fallas de fabricación. La garantía no cubre golpes, humedad ni daños por mal uso.


In [16]:
# ============================================================
# CHALLENGE - PASO 7. COMPARAR SIN RAG VS. CON RAG
# ============================================================
#
# Esta comparación es una parte importante del Challenge.
#
# La pregunta es la misma.
# Lo que cambia es que la segunda respuesta recibe el fragmento
# recuperado de nuestras políticas.

print("=" * 70)
print("PREGUNTA")
print(pregunta_challenge)

print("\n" + "=" * 70)
print("RESPUESTA SIN RAG")
print(respuesta_sin_rag.choices[0].message.content)

print("\n" + "=" * 70)
print("FRAGMENTO RECUPERADO")
print(fragmento_relevante)

print("\n" + "=" * 70)
print("RESPUESTA CON RAG")
print(respuesta_con_rag.choices[0].message.content)


PREGUNTA
Mi producto electrónico dejó de funcionar después de dos meses, ¿qué puedo hacer?

RESPUESTA SIN RAG
¡Lo sentimos mucho!  
1️⃣ Contacta a nuestro servicio de atención al cliente con tu número de pedido y el comprobante de compra.  
2️⃣ Si el producto está dentro del periodo de garantía, lo enviamos para reparación o reemplazo sin cargo.  
3️⃣ Si ya supera la garantía, te ofrecemos un descuento especial o un crédito para una nueva compra.  

Nuestro objetivo es resolverlo lo antes posible. ¡Gracias por confiar en nosotros!

FRAGMENTO RECUPERADO
GARANTÍA: Los productos electrónicos cuentan con 6 meses de garantía por fallas de fabricación. La garantía no cubre golpes, humedad ni daños por mal uso.

RESPUESTA CON RAG
Tu producto está dentro del plazo de garantía de 6 meses por fallas de fabricación. La garantía no cubre golpes, humedad ni daños por mal uso.


## **¿Qué debe observar el alumno?**

La comparación no consiste únicamente en decidir cuál respuesta “suena mejor”.

Debe revisar:

- si la respuesta **sin RAG** menciona condiciones que no aparecen en las políticas;
- si el retrieval seleccionó el fragmento realmente relacionado con la pregunta;
- si la respuesta **con RAG** utiliza la información de la política recuperada;
- si el modelo reconoce cuando el documento no contiene información suficiente.

En este ejemplo, una pregunta sobre un producto electrónico que falla después de dos meses debería recuperar el fragmento de **GARANTÍA**, porque es el que contiene información semánticamente relacionada con la consulta.


## **Mini ejercicio para explicarlo a los alumnos**

Después de ejecutar el ejemplo, puedes pedirles:

1. Cambiar una de las políticas por una política propia.
2. Formular una nueva pregunta.
3. Ejecutar nuevamente los embeddings y el retrieval.
4. Revisar qué fragmento obtuvo la similitud coseno más alta.
5. Comparar la respuesta sin RAG y con RAG.
6. Explicar con sus propias palabras por qué cambió la respuesta.

### Preguntas de comprobación

- ¿Qué hace `SentenceTransformer.encode()`?
- ¿Qué representa un embedding?
- ¿Qué compara la similitud coseno?
- ¿Para qué sirve `np.argmax()`?
- ¿Qué información se agrega al prompt cuando usamos RAG?
- ¿Cuál es la diferencia entre mejorar un prompt y proporcionar contexto externo mediante RAG?

### Relación con el Challenge oficial

Este ejemplo trabaja exactamente las competencias descritas en el enunciado compartido: **generación de embeddings, recuperación mediante similitud coseno y prompting con contexto para contrastar la respuesta con RAG frente a la respuesta sin RAG**.

Para la entrega real, el alumnado debe seguir las instrucciones específicas que aparezcan en la sección oficial del Challenge y mantener completa la sección de código trabajada durante la clase.
